## Practical exercise

The codes of block below:

* Read the data from the file data.csv
* Define a pipeline that includes only a clustering step with Kmeans
* Perform a grid search over the number of clusters using the silhoutte score as target
* Evaluate the silhoutte score when the model with the optimal number of clusters is adopted

The performances are not satisfactory. A possible reason is that the input features are highly correlated and different features span widely different ranges. Implement changes in the pipeline to overcome these problems. Motivate your strategy.

In [1]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClusterMixin
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

In [2]:
def clustering_silhouette_scorer(estimator, X, y=None):
    labels = estimator.predict(X)
    if isinstance(estimator, Pipeline) and len(estimator.steps) > 1:
        X_for_score = estimator[:-1].transform(X)
    else:
        X_for_score = X
    n_labels = len(set(labels))
    if n_labels < 2 or n_labels >= len(labels):
        return -1.0
    return silhouette_score(X_for_score, labels)

X = pd.read_csv('data.csv')

pipeline = Pipeline([
    ("kmeans", KMeans(random_state=RANDOM_STATE, n_init=20)),
])
param_grid = {
    "kmeans__n_clusters": [2, 3, 4, 5, 6, 7],
}
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=clustering_silhouette_scorer,
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)

grid_search.fit(X)

print("Best parameters:", grid_search.best_params_)
print("Best mean CV silhouette score:",grid_search.best_score_)

Best parameters: {'kmeans__n_clusters': 4}
Best mean CV silhouette score: 0.5892773055977691


In [3]:
best_pipeline = grid_search.best_estimator_
cluster_labels = best_pipeline.named_steps["kmeans"].labels_
if isinstance(best_pipeline, Pipeline) and len(best_pipeline.steps) > 1:
    X_for_score = best_pipeline[:-1].transform(X)
else:
    X_for_score = X
final_silhouette = silhouette_score(X_for_score, cluster_labels)
print("Final silhouette score on full data:", final_silhouette)

Final silhouette score on full data: 0.5890675481295585


### Possible solution

In [4]:
def clustering_silhouette_scorer(estimator, X, y=None):
    labels = estimator.predict(X)
    if isinstance(estimator, Pipeline) and len(estimator.steps) > 1:
        X_for_score = estimator[:-1].transform(X)
    else:
        X_for_score = X
    n_labels = len(set(labels))
    if n_labels < 2 or n_labels >= len(labels):
        return -1.0
    return silhouette_score(X_for_score, labels)

X = pd.read_csv('data.csv')

pipeline = Pipeline([
    ("scale", StandardScaler()), # CHANGE 1
    ("pca", PCA(random_state=RANDOM_STATE)), # CHANGE 2
    ("kmeans", KMeans(random_state=RANDOM_STATE, n_init=20)),
])
    
param_grid = {
    "pca__n_components": [2, 3, 5, 8, 12, 20], # CHANGE 3
    "kmeans__n_clusters": [2, 3, 4, 5, 6, 7],
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=clustering_silhouette_scorer,
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)

grid_search.fit(X)

print("Best parameters:", grid_search.best_params_)
print("Best mean CV silhouette score:",grid_search.best_score_)

Best parameters: {'kmeans__n_clusters': 4, 'pca__n_components': 2}
Best mean CV silhouette score: 0.7621110129612461


In [5]:
best_pipeline = grid_search.best_estimator_
cluster_labels = best_pipeline.named_steps["kmeans"].labels_
if isinstance(best_pipeline, Pipeline) and len(best_pipeline.steps) > 1:
    X_for_score = best_pipeline[:-1].transform(X)
else:
    X_for_score = X
final_silhouette = silhouette_score(X_for_score, cluster_labels)
print("Final silhouette score on full data:", final_silhouette)

Final silhouette score on full data: 0.7618738234408879


## Multiple choice questions

## Open questions

### 1. Discuss the Elbow Method and Silhouette Score for selecting the number of clusters. Compare their advantages and limitations.

### 2. Explain the logic of decision trees for classification and regression. Why are they considered interpretable models?